# AIF Databases with yabadaba

This Notebook provides an overview of the database features that yabadaba provides for AIF records.  It builds on features mentioned in the previous Notebook for the Record objects.

## 1. Database overview

### 1.1. Database styles

With yabadaba, data can be accessed and stored in different databases and different types of databases.  The different available database styles are

- 'local' which interacts with a directory of JSON or XML files on your computer plus csv "cache" files to improve query performance.
- 'mongo' which interacts with a MongoDB.  This offers noticeable performance boosts when the number of records is > ~10,000.
- 'cdcs' which interacts with a CDCS/MDCS curator.  This allows for the data to be stored in an open public database.
- 'aif' which behaves like 'local' except that the directory contains AIF files.  This style is part of the yabadaba_aif package.

### 1.2. Database structure

An example database is included here for the demonstration in the aif_database directory.  It consists of subdirectories aif and aif2, and csv files aif.csv and aif2.csv.  The two subdirectories contain identical AIF records and only exist separately as each directory is associated with the record style of the corresponding name.

In [1]:
# Import core yabadaba package which contains the main interaction capabilities
import yabadaba

# Import the AIF-specific subclasses
import yabadaba_aif

## 2. Initializing a database connection

Database interactions can be initialized by specifying a style, a host (location), and any other required access parameters.  Optionally, yabadaba allows for database access settings to be saved and later retrieved using a simple string name (https://github.com/lmhale99/yabadaba/blob/main/doc/3.%20Save%20settings%20across%20sessions.ipynb). 

In [2]:
# Initialize an aif-style database hosted in the aif_database folder
database = yabadaba.load_database(style='aif', host='aif_database')

## 3. Get matching records

Database objects provide a few different methods for querying records. 

- get_records() retrieves all matching records.  If return_df=True is added, it will also return a pandas DataFrame of all of the matching record's metadata fields.
- get_records_df() is the same as above with return_df=True, except the record objects are not returned.
- get_record() retrieves a single matching record if one uniquely exists.  An error is thrown if no matches or multiple matches are found.
- count_records() returns only a count of matching records.

All of these methods also recognize query terms associated with the defined values.  Typically, if it is a term that appears in the metadata() representation it can be used to filter results.

In [3]:
# Show counts of the records of the two "aif" styles
print(database.count_records('aif'))
print(database.count_records('aif2'))

24
24


In [4]:
# Get all records in the AIF2 Record format with adsorptive Methane. Also return the DataFrame representation.
r, df = database.get_records('aif2', exptl_adsorptive_name='Methane', return_df=True)

# Remove any empty DataFrame columns
df.dropna(axis=1, how='all')

,name,exptl_adsorptive_name,exptl_date,exptl_isotherm_type,exptl_temperature,adsnt_material_id,simltn_forcefield_adsorbent,simltn_forcefield_adsorptive,simltn_adsorbent_filename,units_loading,units_pressure,units_temperature,units_fugacity,units_qst
0,CH4_CuBTC_300K,Methane,2023-08-09,absolute,300,CuBTC,see adsorbent file,TraPPE CH4; 12ang linear-force shift,data.CuBTC_CaleroIV_rep111,MOL-PER-KiloGM,BAR,K,BAR,KiloJ-PER-MOL
1,CH4_CuBTC_350K,Methane,2023-08-08,absolute,350,CuBTC,see adsorbent file,TraPPE CH4; 12ang linear-force shift,data.CuBTC_CaleroIV_rep111,MOL-PER-KiloGM,BAR,K,BAR,KiloJ-PER-MOL
2,CH4_IRMOF1_300K,Methane,2023-08-08,absolute,300,IRMOF-1,see adsorbent file,TraPPE CH4; 12ang linear-force shift,data.IRMOF1_mCBAC_rep111,MOL-PER-KiloGM,BAR,K,BAR,KiloJ-PER-MOL
3,CH4_IRMOF1_350K,Methane,2023-08-09,absolute,350,IRMOF-1,see adsorbent file,TraPPE CH4; 12ang linear-force shift,data.IRMOF1_mCBAC_rep111,MOL-PER-KiloGM,BAR,K,BAR,KiloJ-PER-MOL
4,CH4_UiO66_300K,Methane,2023-08-16,absolute,300,UiO-66,see adsorbent file,TraPPE CH4; 12ang linear-force shift,data.UiO66_Snurr_rep222,MOL-PER-KiloGM,BAR,K,BAR,KiloJ-PER-MOL
5,CH4_UiO66_350K,Methane,2023-08-18,absolute,350,UiO-66,see adsorbent file,TraPPE CH4; 12ang linear-force shift,data.UiO66_Snurr_rep222,MOL-PER-KiloGM,BAR,K,BAR,KiloJ-PER-MOL
6,CH4_ZIF8_300K,Methane,2023-08-07,absolute,300,ZIF-8,see adsorbent file,TraPPE CH4; 12ang linear-force shift,data.ZIF8_Snurr_rep222,MOL-PER-KiloGM,BAR,K,BAR,KiloJ-PER-MOL
7,CH4_ZIF8_350K,Methane,2023-08-09,absolute,350,ZIF-8,see adsorbent file,TraPPE CH4; 12ang linear-force shift,data.ZIF8_Snurr_rep222,MOL-PER-KiloGM,BAR,K,BAR,KiloJ-PER-MOL


Additionally, the returned records are in a numpy array allowing for numpy-style indexing.  This allows for r and df to be used in conjunction to do further parsing and sorting after getting the records.

In [5]:
# Filter the records further based on temperature
r_300K = r[df.exptl_temperature == 300]

for record in r_300K:
    print(record.name)

CH4_CuBTC_300K
CH4_IRMOF1_300K
CH4_UiO66_300K
CH4_ZIF8_300K


## 4. Other database operations

### 4.1. Record-based operations

- add_record() adds a new record to the database.
- update_record() updates the contents of a record already in the database.
- delete_record() deletes a record from the database.
- copy_records() copies multiple records from one database to another.
- destroy_records() deletes multiple records and associated tars from the database.

### 4.2 Tar-base operations

yabadaba also supports storing raw data files in the database associated with different records.  This is done by placing all associated files inside a tar.gz file.
- get_tar() retrieves a tar file associated with a record.
- add_tar() adds a tar file to the database for a record.
- update_tar() replaces a record's tar file in the database.
- delete_tar() deletes a record's tar file.

If this is of interest, the Record objects also have additional features making file retrieval from the tars more convenient...

## Time trials

- aif (style 1) loads loop data as arrays
- aif2 (style 2) loads loop data as pandas dataframes.
- get_records() with a query term loads the csv then pre-filters before loading only matching records
- get_records() without a query term loads the csv then loads all records
- Path loop with load_record skips the initial csv loading

In this simple example
- Loading the csv ~29 ms
- Loading all records with aif ~39 ms
- Loading all records with aif2 ~63 ms
- Using the built-in query is fastest here as loading the csv < loading all records and only the 2 matching records get loaded.

Should test against 1000s of records to compare scalibility of csv file and record loading speeds.  Ideally, 10,000+ should switch to a Mongo backend. 

In [26]:
%%timeit
records = database.get_records('aif', exptl_adsorptive_name='Water')
len(records)

32.5 ms ± 1.04 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [27]:
%%timeit
records = database.get_records('aif2', exptl_adsorptive_name='Water')
len(records)

35.7 ms ± 1.12 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [28]:
%%timeit
records = []
allrecords = database.get_records('aif')
for record in allrecords:
    if record.exptl_adsorptive_name=='Water':
        records.append(record)
len(records)

68.7 ms ± 1.25 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [29]:
%%timeit
records = []
allrecords = database.get_records('aif2')
for record in allrecords:
    if record.exptl_adsorptive_name=='Water':
        records.append(record)
len(records)

92.3 ms ± 1.22 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [32]:
%%timeit
records = []
for recordpath in Path(database.host, 'aif').glob('*.aif'):
    record = yabadaba.load_record('aif', aif=recordpath)
    if record.exptl_adsorptive_name=='Water':
        records.append(record)
len(records)

39.5 ms ± 1.09 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [31]:
%%timeit
records = []
for recordpath in Path(database.host, 'aif2').glob('*.aif'):
    record = yabadaba.load_record('aif2', aif=recordpath)
    if record.exptl_adsorptive_name=='Water':
        records.append(record)
len(records)

63.8 ms ± 791 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
